# Figures

Every figure from the paper. Each section loads a pre-computed intermediate CSV (produced by
the corresponding `evaluate/compute_*.py` script) from `PATH_ANALYSIS` (`evaluate/data/` by
default, see `src/config.py`) and plots it.

Run this notebook from its own directory (`evaluate/`).

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root (this notebook lives in evaluate/, alongside its data files)

import json
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.lines import Line2D

from src.config import PATH_ANALYSIS
from evaluate.plotting_config import VARIATION_MAPPINGS, MODEL_MAPPINGS

os.makedirs("figures", exist_ok=True)

In [ ]:
df = pd.read_csv(PATH_ANALYSIS / 'marginal_action_likelihoods.csv')

In [ ]:
print(list(MODEL_MAPPINGS.keys()))

# Figure 1: base preferences

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "legend.frameon": False,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm",
    "mathtext.rm": "serif",
})

# ------------------------------------------------------------------
# 1. Define group structure + spacing
# ------------------------------------------------------------------
group_sizes = [4, 4, 4, 3, 3, 4]
gap = 0.5

xpos = []
current = 0.0
for g in group_sizes:
    xpos.extend(current + np.arange(g))
    current += g + gap

xpos = np.array(xpos)

# ------------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------------
plot_df = df.copy()
# family-grouped order: llama, mistral, qwen, deepseek, anthropic, openai (matches MODEL_MAPPINGS)
order = [m for m in MODEL_MAPPINGS.keys() if m in plot_df["model_id"].unique()]
palette = [MODEL_MAPPINGS.get(mid, {}).get("display_color", "grey") for mid in order]

# ------------------------------------------------------------------
# 3. Plot manually with proper spacing
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(15, 5))

box_width = 0.1

for i, model_id in enumerate(order):
    data = plot_df[plot_df["model_id"] == model_id]["p_action2_base"].values
    
    parts = ax.violinplot(
        [data],
        positions=[xpos[i]],
        widths=0.85,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        #bw_method=0.1
    )
    
    # Color the violin
    for pc in parts['bodies']:
        pc.set_facecolor(palette[i])
        pc.set_alpha(0.95)
        pc.set_edgecolor('black')
        pc.set_linewidth(2)
    
    # Calculate statistics
    q1, median, q3 = np.percentile(data, [25, 50, 75])
    iqr = q3 - q1
    
    box = plt.Rectangle(
        (xpos[i] - box_width/2, q1),
        box_width,
        q3 - q1,
        facecolor='black',
        edgecolor='black',
        linewidth=2,
        zorder=4
    )
    ax.add_patch(box)
    
    ax.scatter(
        xpos[i],
        median,
        s=25,
        marker="o",
        facecolors="white",
        edgecolors="black",
        linewidths=1.1,
        zorder=5
    )

# ------------------------------------------------------------------
# 4. Axes, ticks, labels
# ------------------------------------------------------------------
ax.set_xticks(xpos)
ax.set_xticklabels(
    [MODEL_MAPPINGS.get(mid, {}).get("display_name", str(mid)) for mid in order],
    rotation=45,
    ha="right"
)

ax.set_ylabel("Marginal Action Likelihood \n P(rule-violating action)", labelpad=8)
ax.set_xlabel("")
ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
ax.set_ylim(-0.05, 1.05)

ax.axhline(0.5, linestyle="--", linewidth=2, color="black", alpha=0.7, zorder=1)

# ------------------------------------------------------------------
# 5. Visual separators between model families
# ------------------------------------------------------------------
boundaries = np.cumsum(group_sizes)[:-1]
sep_positions = []

for boundary_idx in boundaries:
    # Position separator in the middle of the gap
    sep_pos = (xpos[boundary_idx - 1] + xpos[boundary_idx]) / 2
    sep_positions.append(sep_pos)
    ax.axvline(sep_pos, color='black', linestyle='-', linewidth=2, alpha=0.35)

# ------------------------------------------------------------------
# 6. Final styling
# ------------------------------------------------------------------
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(axis="both", which="major", direction="out", length=6, width=1.5)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=1)

ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=False)

ax.axhspan(-0.5, 0.25, color="#f4cccc", alpha=0.3, zorder=-3)

# Set x-axis limits with some padding
ax.set_xlim(xpos[0] - 0.75, xpos[-1] + 0.75)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout(pad=0.7)
#plt.savefig("Figures/llm_base_mal_violin.pdf", bbox_inches='tight')
plt.show()

# Figure 2: CPS scores

#### original code: sauter_metrics.ipynb

In [ ]:
df_statistics = pd.read_csv(PATH_ANALYSIS / "cps_statistics.csv")

df_statistics_avg = pd.read_csv(PATH_ANALYSIS / "cps_statistics_avg.csv")

In [ ]:
df_statistics_avg

In [ ]:
sns.set_theme(style="ticks", font_scale=1.0)  # reduced from 1.2
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 14,        # reduced from 19
    "legend.frameon": False,
    "xtick.labelsize": 13,       # reduced from 19
    "ytick.labelsize": 12,       # reduced from 17
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

fig, axes = plt.subplots(1, 3, figsize=(11, 7.1), sharex=False, sharey=True)

variations = ["Consequentialist", "Emotional", "Relational"]

for ax, variation in zip(axes, variations):

    sub = df_statistics[df_statistics["variation"] == variation].copy()
    # family-grouped order, llama at top to openai at bottom (matches MODEL_MAPPINGS, reversed)
    model_order = [m for m in MODEL_MAPPINGS.keys() if m in sub["model_id"].values][::-1]
    sub = sub.set_index("model_id").loc[model_order]

    gap_after = [3, 6, 9, 13, 17]
    gap_size = 0.3
    ypos = []
    current_y = 0.0

    for i in range(len(sub)):
        ypos.append(current_y)
        current_y += 1.0
        if i in gap_after:
            current_y += gap_size

    ypos = np.array(ypos)
    ax.set_yticks(ypos)
    ax.set_yticklabels([MODEL_MAPPINGS[m]["display_name"] for m in sub.index], fontsize=12)

    ax.axvline(0, linestyle='--', color='gray', linewidth=2)

    for i, ((model_id, row), y) in enumerate(zip(sub.iterrows(), ypos)):
        color = MODEL_MAPPINGS[model_id]["display_color"]
        ax.errorbar(
            x=row["ci_mean"], y=y,
            xerr=[[row["ci_mean"] - row["ci_lower"]],
                  [row["ci_upper"] - row["ci_mean"]]],
            fmt='o',
            markersize=8,
            ecolor=color,
            color=color,
            elinewidth=2.5,
        )


    # add average row across models for the current variation
    avg_row = df_statistics_avg.set_index("variation").loc[variation]

    y_avg = -1.3

    ax.errorbar(
        x=avg_row["ci_mean"], y=y_avg,
        xerr=[[avg_row["ci_mean"] - avg_row["ci_lower"]],
              [avg_row["ci_upper"] - avg_row["ci_mean"]]],
        fmt='o',
        markersize=8,
        ecolor='#6B6B6B',
        color='#6B6B6B',
        elinewidth=2.5,
    )

    ax.set_yticks(list(ypos) + [y_avg])
    labels = ax.set_yticklabels(
        [MODEL_MAPPINGS[m]["display_name"] for m in sub.index] + ["Average"],
        fontsize=12,
    )
    if labels:
        labels[-1].set_fontweight('bold')
    ax.set_ylim(-2.2, max(ypos) + 0.6)
    ax.tick_params(axis="both", which="major", direction="out", length=5, width=1.5)
    ax.tick_params(axis="both", which="minor", direction="in", length=3, width=1)

    ax.text(0.085, max(ypos) + 2.0, variation, fontsize=15, fontweight='bold',
            color=VARIATION_MAPPINGS[variation]['display_color'],
            ha='center', va='center', transform=ax.transData)

    separators = [ypos[3] + 0.55, ypos[6] + 0.55, ypos[9] + 0.55, ypos[13] + 0.55, ypos[17] + 0.55, y_avg + 0.55]
    for sep in separators:
        ax.axhline(y=sep, color='black', linestyle='-', linewidth=2, alpha=0.35)

    ax.grid(axis="x", linestyle='-', linewidth=0.6, alpha=0.5)
    ax.set_xlim(-0.01, 0.17)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    color = VARIATION_MAPPINGS[variation]['display_color']
    var_letter = variation[0].upper()

    # X-axis label block
    lab_y = -4.5
    ax.text(0.06,   lab_y,        "CPS", size=15, weight="bold", ha="right", transform=ax.transData)
    ax.text(0.06,   lab_y + 0.4,  "(", size=11, ha="left", transform=ax.transData)
    ax.text(0.071,  lab_y + 0.4,  ")", size=11, ha="left", transform=ax.transData)
    ax.text(0.0635, lab_y + 0.35, var_letter, size=11, weight="bold", color=color, ha="left", transform=ax.transData)
    ax.text(0.08,   lab_y + 0.1,  "(95% CI)", size=15, ha="left", transform=ax.transData)

plt.tight_layout(pad=1.0)
plt.savefig("figures/cps_plot.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_statistics

# Figure 3: Boundary mass vs flip rate

#### original code: llm_main_analysis.ipynb

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})


plot_df = df_statistics.copy()
titles = ["Consequentialist", "Emotional", "Relational"]

# --- Create Subplots ---
# Increase figure height slightly to accommodate the legend at the bottom
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True, constrained_layout=True)

# --- 0) helpers ---
def get_style(model_id):
    info = MODEL_MAPPINGS.get(model_id, None)
    if info is None:
        return dict(label=model_id, color="grey", marker="o")
    return dict(label=info["display_name"],
                color=info["display_color"],
                marker=info["marker"])

# --- 1) certainty -> marker size (your current approach) ---
# mae_min, mae_max = df_statistics["mae_base"].min(), df_statistics["mae_base"].max()
# df_statistics["mae_norm"] = (df_statistics["mae_base"] - mae_min) / (mae_max - mae_min + 1e-12)
# df_statistics["certainty"] = 1 - df_statistics["mae_norm"]

# MIN_S, MAX_S = 100, 250
# df_statistics["msize"] = MIN_S + df_statistics["certainty"] * (MAX_S - MIN_S)

# --- 2) plot: 3 small multiples ---
variations = list(VARIATION_MAPPINGS.keys())


for i, (ax, var) in enumerate(zip(axes, variations)):
    sub = plot_df[plot_df["variation"] == var].copy()

    for model_id in sub["model_id"].unique():
        st = get_style(model_id)
        ss = sub[sub["model_id"] == model_id]

        ax.scatter(
            ss["boundary_mass"],
            ss["flip_rate"],
            s=225,
            #s=ss["msize"],
            alpha=0.9,
            c=st["color"],
            marker=st["marker"],
            edgecolors="black",
            linewidths=1.0,
            zorder=4
        )

    #ax.axvline(0.05, linestyle="--", linewidth=1)
    #ax.axhline(0.0, linestyle="--", linewidth=1)

    ax.set_title(titles[i], fontsize=26, pad=12, color=VARIATION_MAPPINGS[titles[i]]['display_color'])


    #ax.set_xlabel("Boundary Mass", labelpad=10)
    ax.text(0.17, 0.025, "BM", size=22, weight="bold", ha="right", transform=ax.transData)
    ax.text(0.1875, 0.0215, "0.1", size=16, weight="bold", ha="right", transform=ax.transData)
    # ax.text(0.06, -3.2+3.0, "CPS", size=22, weight="bold", ha="right", transform=ax.transData)
    # # brackets for superscript
    # ax.text(0.06, -2.85+3.0, "(", size=16, ha="left", transform=ax.transData)
    # ax.text(0.071, -2.85+3.0, ")", size=16, ha="left", transform=ax.transData)
    # # Superscript
    # ax.text(0.0635, -2.9+3.0, var_letter, size=16, weight="bold", color=color, ha="left", transform=ax.transData)

    if i == 0:
        ax.set_ylabel("Flip Rate", labelpad=10)

    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlim(0.01, 0.325)


meta_models = ['meta_llama-2-7b-chat', 'meta_llama-3-8B-instruct', 'meta_llama-3.1-8B-instruct', 'meta_llama-3.1-70b-instruct']
mistral_models = ['mistral_mixtral-8x7b-instruct_8bit', 'mistral_mistral-7b-instruct-v0.1', 'huggingfaceh4_zephyr-7b-beta', 'teknium_openhermes-2.5-mistral-7b']
qwen_models = ['qwen_qwen1.5-7b-chat', 'qwen_qwen2-7b-instruct', 'qwen_qwen3-4b-instruct', 'qwen_qwen3-8b']
deepseek_models = ['deepseek_deepseek-llm-7b-chat', 'deepseek-ai_DeepSeek-V3', 'deepseek-ai_DeepSeek-V3.1']
anthropic_models = ['claude_claude-3-haiku-20240307', 'claude_claude-haiku-4-5-20251001', 'claude_claude-sonnet-4-5-20250929']
openai_models = ['openai_gpt-4o-mini', 'openai_gpt-4.1', 'openai_gpt-4.1-mini', 'openai_gpt-5.1']

# --- 1. Custom Layout Configuration ---
column_x_positions = [0.00, 0.185, 0.405, 0.565, 0.76, 0.915] 

y_header = 1.05         # Vertical start (top of the legend box)
y_header_offset = 0.15 # Space between "Meta" and the first model
row_height = 0.18      # Vertical gap between model rows

# [left, bottom, width, height]
leg_ax = fig.add_axes([0.05, -0.35, 0.9, 0.3]) 
leg_ax.set_xlim(0, 1)
leg_ax.set_ylim(0, 1)
leg_ax.axis('off')

# --- 2. Data Mapping ---
company_map = {
    "Meta": meta_models,
    "Mistral": mistral_models,
    "Qwen": qwen_models,
    "DeepSeek": deepseek_models,
    "Anthropic": anthropic_models,
    "OpenAI": openai_models
}

# --- 3. Manual Drawing Loop ---
for col_idx, (company, models) in enumerate(company_map.items()):
    # Get the manual X position for this specific column
    curr_x = column_x_positions[col_idx]
    
    
    for row_idx, mid in enumerate(models):
        curr_y = y_header - y_header_offset - (row_idx * row_height)
        
        if mid not in MODEL_MAPPINGS:
            continue
            
        m_info = MODEL_MAPPINGS[mid]
        
        leg_ax.scatter(curr_x + 0.01, curr_y, 
                       marker=m_info['marker'], 
                       color=m_info['display_color'], 
                       edgecolor='black', s=200, linewidth=0.8, 
                       clip_on=False, zorder=10)
        
        leg_ax.text(curr_x + 0.0225, curr_y, m_info['display_name'], 
                    va='center', ha='left', fontsize=18)

plt.tight_layout()
#plt.savefig("Figures/bm_fr.pdf", bbox_inches="tight", dpi=300)
plt.show()

# Refusal/ invalid plot

#### original code: llm_analysis.ipynb

In [ ]:
df_refusal_invalid = pd.read_csv(PATH_ANALYSIS / "refusal_invalid_stats.csv")

In [ ]:
# sort dataframes by model_id to ensure consistent ordering
order = [model for model in MODEL_MAPPINGS.keys()]
sorterIndex = dict(zip(order, range(len(order))))
df_refusal_invalid['model_id_rank'] = df_refusal_invalid['model_id'].map(sorterIndex)
df_refusal_invalid.sort_values(['model_id_rank'], ascending = [True], inplace = True)
df_refusal_invalid.drop(labels='model_id_rank', axis=1, inplace=True)

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 17,
    "legend.frameon": False,
    "xtick.labelsize": 16,
    "ytick.labelsize": 12,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,

    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
    
})

# Extract model names and proportions
models = df_refusal_invalid['model_id'].tolist()

refusals = [v["refusal_proportion"] for _, v in df_refusal_invalid.iterrows()]
invalids = [v["invalid_proportion"] for _, v in df_refusal_invalid.iterrows()]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, refusals, width, label='Refusals', edgecolor="black", color='#8E3B3B')
ax.bar(x + width/2, invalids, width, label='Invalid Responses', edgecolor="black", color='#C9B458')

# Add value labels
#for i, (bm, vm) in enumerate(zip(refusals, invalids)):
    #ax.text(x[i] - width/2+0.05, bm + 0.002, f"{bm:.3f}", ha="center", va="bottom", fontsize=9, fontweight='bold', rotation=90)
    #ax.text(x[i] + width/2+0.05, vm + 0.002, f"{vm:.3f}", ha="center", va="bottom", fontsize=9, fontweight='bold', rotation=90)

ax.set_ylabel('Proportion')
ax.set_xticks(x)
ax.set_xticklabels([MODEL_MAPPINGS[m]['display_name'] for m in models], rotation=45, ha='right', fontsize=12)
ax.legend(fontsize=12)

# Add grid
ax.grid(True, alpha=0.25, axis='y', linestyle='-', linewidth=0.8)
ax.set_axisbelow(True)


sns.despine()
plt.tight_layout()
plt.savefig("figures/llm_refusals_invalids.pdf", dpi=300)
plt.show()

# CPS distribution plot

#### original code: cps_distribution_plot.ipynb

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

# Ordered list of model IDs as they appear in MODEL_MAPPINGS
model_ids = list(MODEL_MAPPINGS.keys())
n_models = len(model_ids)

data_all = df.copy()
x_min = data_all['CPS'].min() - 0.05
x_max = data_all['CPS'].max() + 0.05

# Tall figure to accommodate 22 rows
fig = plt.figure(figsize=(20, n_models * 0.9), dpi=300)
outer_gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05)

for col_idx, variation_name in enumerate(variations):
    inner_gs = gridspec.GridSpecFromSubplotSpec(
        n_models, 1,
        subplot_spec=outer_gs[col_idx],
        hspace=-0.3
    )
    axs = [fig.add_subplot(inner_gs[i]) for i in range(n_models)]

    data = data_all[data_all['variation'] == variation_name]

    for i, model_id in enumerate(model_ids):
        ax = axs[i]
        subset = data[data['model_id'] == model_id]['CPS']
        color = mcolors.to_rgb(MODEL_MAPPINGS[model_id]['display_color'])

        if len(subset) > 1:
            # --- Split KDE: positive side (full color) and negative side (muted) ---
            sns.kdeplot(
                subset, ax=ax,
                bw_adjust=0.4,
                fill=True, alpha=1.0,
                color=color,
                linewidth=0,
                clip=(x_min, 0)          # left of zero: muted
            )
            sns.kdeplot(
                subset, ax=ax,
                bw_adjust=0.4,
                fill=True, alpha=1.0,
                color=color,
                linewidth=0,
                clip=(0, x_max)          # right of zero: full color
            )
            # --- White outline over the full KDE ---
            sns.kdeplot(
                subset, ax=ax,
                bw_adjust=0.4,
                fill=False,
                color='white',
                linewidth=1.5,
                clip=(x_min, x_max)
            )

            #peak = ax.get_ylim()[1]
            #ax.set_ylim(0, peak * 0.5)

            # --- Mean line ---
            mean_val = subset.mean()
            y_top = ax.get_ylim()[1] * 0.5
            ax.plot(
                [mean_val, mean_val],
                [0, y_top],              # explicitly bounded to this axis's height
                lw=2,
                color='black',
                linestyle='-',
                alpha=0.9,
                zorder=4,
                clip_on=True             # clip to this axis only
            )


        ax.axvline(0, lw=2, color='black', linestyle='--', clip_on=True, zorder=3)

        # Model label on leftmost column only
        if col_idx == 0:
            # Shorten model name for display if needed
            label = MODEL_MAPPINGS[model_id].get('display_name', model_id.split('/')[-1])
            ax.text(
                -0.02, 0.2,
                label,
                fontweight="bold",
                color=color,
                ha="right", va="center",
                transform=ax.transAxes,
                clip_on=False,
                fontsize=14,
                fontfamily="serif"
            )
        ax.set_xticks([-0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])
        ax.patch.set_alpha(0)
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.set_xlabel("")
        ax.set_xlim(x_min, x_max)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)

        if i < n_models - 1:
            ax.set_xticks([])
            ax.spines['bottom'].set_visible(False)
        else:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_color("#000000")
            ax.spines['bottom'].set_linewidth(1.0)
            ax.tick_params(axis='x', labelsize=16)
            ax.set_xlabel("Scenario CPS Distribution", labelpad=8, fontsize=18)

        if i == 0:
            ax.set_title(
                variation_name,
                fontsize=25,
                pad=15,
                color=VARIATION_MAPPINGS[variation_name]['display_color']
            )

plt.tight_layout()
#plt.savefig("Figures/cps_distributions.pdf", bbox_inches='tight', dpi=300)
plt.show()

# Base MAL vs Var MAL

#### original code: llm_main_analysis.ipynb

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

plot_df = df.copy()
plot_df = plot_df.groupby(['model_id', 'variation']).apply(lambda x: x[['p_action2_base', 'p_action2_variation']].mean()).reset_index()

titles = ["Consequentialist", "Emotional", "Relational"]

# --- Create Subplots ---
# Increase figure height slightly to accommodate the legend at the bottom
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True, constrained_layout=True)

# --- 0) helpers ---
def get_style(model_id):
    info = MODEL_MAPPINGS.get(model_id, None)
    if info is None:
        return dict(label=model_id, color="grey", marker="o")
    return dict(label=info["display_name"],
                color=info["display_color"],
                marker=info["marker"])

# --- 1) certainty -> marker size (your current approach) ---
# mae_min, mae_max = agg["mae_base"].min(), agg["mae_base"].max()
# agg["mae_norm"] = (agg["mae_base"] - mae_min) / (mae_max - mae_min + 1e-12)
# agg["certainty"] = 1 - agg["mae_norm"]

# MIN_S, MAX_S = 100, 250
# agg["msize"] = MIN_S + agg["certainty"] * (MAX_S - MIN_S)

# --- 2) plot: 3 small multiples ---
variations = list(VARIATION_MAPPINGS.keys())

for i, (ax, var) in enumerate(zip(axes, variations)):
    sub = plot_df[plot_df["variation"] == var].copy()

    for model_id in sub["model_id"].unique():
        st = get_style(model_id)
        ss = sub[sub["model_id"] == model_id]

        ax.scatter(
            ss["p_action2_base"],
            ss["p_action2_variation"],
            s=225,
            #s=ss["msize"],
            alpha=0.9,
            c=st["color"],
            marker=st["marker"],
            edgecolors="black",
            linewidths=1.0,
            zorder=4
        )


    ax.set_title(titles[i], fontsize=26, pad=12, color=VARIATION_MAPPINGS[titles[i]]['display_color'])


    ax.set_xlabel("Base version\nP(rule-violating action)", labelpad=8, fontsize=20)

    if i == 0:
        ax.set_ylabel("Contextual variation\nP(rule-violating action)", labelpad=8, fontsize=20)

    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlim(0.225, 0.65)
    ax.set_ylim(0.225, 0.65)

    ax.plot([0.225, 0.65], [0.225, 0.65], linestyle="--", color="black", alpha=0.7, zorder=2)

meta_models = ['meta_llama-2-7b-chat', 'meta_llama-3-8B-instruct', 'meta_llama-3.1-8B-instruct', 'meta_llama-3.1-70b-instruct']
mistral_models = ['mistral_mixtral-8x7b-instruct_8bit', 'mistral_mistral-7b-instruct-v0.1', 'huggingfaceh4_zephyr-7b-beta', 'teknium_openhermes-2.5-mistral-7b']
qwen_models = ['qwen_qwen1.5-7b-chat', 'qwen_qwen2-7b-instruct', 'qwen_qwen3-4b-instruct', 'qwen_qwen3-8b']
deepseek_models = ['deepseek_deepseek-llm-7b-chat', 'deepseek-ai_DeepSeek-V3', 'deepseek-ai_DeepSeek-V3.1']
anthropic_models = ['claude_claude-3-haiku-20240307', 'claude_claude-haiku-4-5-20251001', 'claude_claude-sonnet-4-5-20250929']
openai_models = ['openai_gpt-4o-mini', 'openai_gpt-4.1', 'openai_gpt-4.1-mini', 'openai_gpt-5.1']

# --- 1. Custom Layout Configuration ---
column_x_positions = [0.00, 0.185, 0.405, 0.565, 0.76, 0.915] 

y_header = 1.0         # Vertical start (top of the legend box)
y_header_offset = 0.15 # Space between "Meta" and the first model
row_height = 0.18      # Vertical gap between model rows

# Create the canvas (positioned below the main plots)
# [left, bottom, width, height]
leg_ax = fig.add_axes([0.05, -0.35, 0.9, 0.3]) 
leg_ax.set_xlim(0, 1)
leg_ax.set_ylim(0, 1)
leg_ax.axis('off')

# --- 2. Data Mapping ---
company_map = {
    "Meta": meta_models,
    "Mistral": mistral_models,
    "Qwen": qwen_models,
    "DeepSeek": deepseek_models,
    "Anthropic": anthropic_models,
    "OpenAI": openai_models
}

# --- 3. Manual Drawing Loop ---
for col_idx, (company, models) in enumerate(company_map.items()):
    curr_x = column_x_positions[col_idx]
    
    for row_idx, mid in enumerate(models):
        # Calculate Y position moving downwards
        curr_y = y_header - y_header_offset - (row_idx * row_height)
            
        m_info = MODEL_MAPPINGS[mid]
        
        leg_ax.scatter(curr_x + 0.01, curr_y, 
                       marker=m_info['marker'], 
                       color=m_info['display_color'], 
                       edgecolor='black', s=200, linewidth=0.8, 
                       clip_on=False, zorder=10)
        
        leg_ax.text(curr_x + 0.0225, curr_y, m_info['display_name'], 
                    va='center', ha='left', fontsize=18)

plt.tight_layout()
#plt.savefig("Figures/mal_base_mal_var.pdf", bbox_inches="tight", dpi=300)
plt.show()

# Base agreement vs var agreement

#### original code: human_llm_comparison

Needs `human_llm_agreement.csv`, which isn't included in this repo.

In [ ]:
human_agg_df = pd.read_csv(PATH_ANALYSIS / "human_llm_agreement.csv")

In [ ]:
# --- Style ---
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

# --- Prepare Data ---
plot_df = human_agg_df.copy()
titles = ["Consequentialist", "Emotional", "Relational"]

# --- Create Subplots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True, constrained_layout=True)

for i, var in enumerate(titles):
    ax = axes[i]
    var_df = human_agg_df[human_agg_df['variation'] == var]
    
    for model_id in MODEL_MAPPINGS.keys():
        color = MODEL_MAPPINGS[model_id]['display_color']
        ss = var_df[var_df["model_id"] == model_id]

        # Scatter Plot
        ax.scatter(
            ss["base_agreement"],
            ss["variation_agreement"],
            s=180, 
            alpha=0.9,
            c=color,
            marker=MODEL_MAPPINGS[model_id]["marker"],
            edgecolors="black",
            linewidths=1.0,
            zorder=4
        )
    # --- Subplot Config ---
    ax.set_title(titles[i], fontsize=26, pad=12, color=VARIATION_MAPPINGS[titles[i]]['display_color'])
    ax.set_xlabel("Base Agreement Rate", labelpad=10)
    if i == 0:
        ax.set_ylabel("Variation Agreement Rate", labelpad=10)
    
    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0.175, 0.725)
    ax.set_ylim(0.175, 0.725)
    ax.add_line(plt.Line2D([0.175, 0.725], [0.175, 0.725], color='gray', linestyle='--', linewidth=1.5, zorder=1))

meta_models = ['meta_llama-2-7b-chat', 'meta_llama-3-8B-instruct', 'meta_llama-3.1-8B-instruct', 'meta_llama-3.1-70b-instruct']
mistral_models = ['mistral_mixtral-8x7b-instruct_8bit', 'mistral_mistral-7b-instruct-v0.1', 'huggingfaceh4_zephyr-7b-beta', 'teknium_openhermes-2.5-mistral-7b']
qwen_models = ['qwen_qwen1.5-7b-chat', 'qwen_qwen2-7b-instruct', 'qwen_qwen3-4b-instruct', 'qwen_qwen3-8b']
deepseek_models = ['deepseek_deepseek-llm-7b-chat', 'deepseek-ai_DeepSeek-V3', 'deepseek-ai_DeepSeek-V3.1']
anthropic_models = ['claude_claude-3-haiku-20240307', 'claude_claude-haiku-4-5-20251001', 'claude_claude-sonnet-4-5-20250929']
openai_models = ['openai_gpt-4o-mini', 'openai_gpt-4.1', 'openai_gpt-4.1-mini', 'openai_gpt-5.1']

# --- 1. Custom Layout Configuration ---
column_x_positions = [0.00, 0.185, 0.405, 0.565, 0.76, 0.915] 

y_header = 0.85        # Vertical start (top of the legend box)
y_header_offset = 0.15 # Space between "Meta" and the first model
row_height = 0.18      # Vertical gap between model rows

# [left, bottom, width, height]
leg_ax = fig.add_axes([0.05, -0.35, 0.9, 0.3]) 
leg_ax.set_xlim(0, 1)
leg_ax.set_ylim(0, 1)
leg_ax.axis('off')

# --- 2. Data Mapping ---
company_map = {
    "Meta": meta_models,
    "Mistral": mistral_models,
    "Qwen": qwen_models,
    "DeepSeek": deepseek_models,
    "Anthropic": anthropic_models,
    "OpenAI": openai_models
}

# --- 3. Manual Drawing Loop ---
for col_idx, (company, models) in enumerate(company_map.items()):
    curr_x = column_x_positions[col_idx]
    
    for row_idx, mid in enumerate(models):
        curr_y = y_header - y_header_offset - (row_idx * row_height)
        
        if mid not in MODEL_MAPPINGS:
            continue
            
        m_info = MODEL_MAPPINGS[mid]
        
        leg_ax.scatter(curr_x + 0.01, curr_y, 
                       marker=m_info['marker'], 
                       color=m_info['display_color'], 
                       edgecolor='black', s=200, linewidth=0.8, 
                       clip_on=False, zorder=10)
        
        # Draw Label (offset slightly from the marker)
        leg_ax.text(curr_x + 0.0225, curr_y, m_info['display_name'], 
                    va='center', ha='left', fontsize=18)


# --- 4. Final Output ---
# You must use bbox_inches="tight" to capture the legend axis
plt.tight_layout()
# plt.savefig("Figures/human_llm_alignment_subplots.pdf", 
#             bbox_inches="tight", dpi=300)
plt.show()


# Layerwise accuracy plot

#### original code: layer_plot.ipynb

Needs `llama-3.1-8b-instruct_layer_selection_cv.pkl`, produced by `evaluate/analyze_layers.py`
from extracted activations (also not included in this repo). See
`evaluate/data/layer_accuracies_by_variation.pdf` for the rendered plot.

In [ ]:
llama_data = pd.read_pickle(PATH_ANALYSIS / 'llama-3.1-8b-instruct_layer_selection_cv.pkl')

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

picked_layer_dict = {'meta_llama-3.1-8B-instruct': {
        'Consequentialist': 13,
        'Emotional': 14,
        'Relational': 15
}, 'qwen_qwen3-8b': {
    'Consequentialist': 17,
        'Emotional': 19,
        'Relational': 18
}
        }

model_results = [llama_data]

results_data = {
    'Consequentialist': {},
    'Emotional': {},
    'Relational': {}
}

for model_data in model_results:
    model_id = model_data['model_id']
    for variation in ['Consequentialist', 'Emotional', 'Relational']:
        if variation in model_data:
            results_data[variation][model_id] = {
                'mean_accs': model_data[variation]['mean_accs'],
                'std_accs': model_data[variation]['std_accs']
            }

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True, constrained_layout=True)

layers = np.arange(32)
variations = ['Consequentialist', 'Emotional', 'Relational']

# Get all unique models
all_models = [model_data['model_id'] for model_data in model_results]

# Determine y-axis limits based on data
all_means = []
for var_data in results_data.values():
    for model_data in var_data.values():
        all_means.extend(model_data['mean_accs'])
y_min = max(0.5, min(all_means) - 0.05)
y_max = min(1.0, max(all_means) + 0.05)

for i, (ax, var) in enumerate(zip(axes, variations)):
    var_data = results_data[var]
    
    # Plot each model for this variation
    for model_id in all_models:
        if model_id not in var_data:
            continue
            
        result = var_data[model_id]
        mean_accs = np.array(result['mean_accs'])
        std_accs = np.array(result['std_accs'])
        
        # Get model style
        if model_id in MODEL_MAPPINGS:
            style = MODEL_MAPPINGS[model_id]
            color = style['display_color']
            marker = style['marker']
        else:
            color = 'gray'
            marker = 'o'
        
        # Plot mean line
        ax.plot(layers, mean_accs, 
                color=color, 
                linewidth=2.5, 
                alpha=1.0,
                zorder=3)
        
        # Plot shaded std region
        ax.fill_between(layers, 
                         mean_accs - std_accs, 
                         mean_accs + std_accs,
                         color=color, 
                         alpha=0.2,
                         zorder=2)
        
        # ===== HIGHLIGHT BEST LAYER =====
        best_layer_idx = np.argmax(mean_accs)
        best_layer = layers[best_layer_idx]
        best_acc = mean_accs[best_layer_idx]
        
        ax.scatter(best_layer, best_acc,
                   marker='^',
                   s=200,
                   color=color,
                   edgecolors='black',
                   linewidths=1.5,
                   zorder=5,
                   alpha=0.75)
        
        

        x = picked_layer_dict[model_id][var]
        y = mean_accs[x] 

        ax.scatter(x, y,
                marker='*',  
                s=260,
                color=color,
                edgecolors='black',
                linewidths=1.5,
                zorder=6)
    
    # Styling
    ax.set_title(var, fontsize=26, pad=12, 
                 color=VARIATION_MAPPINGS[var]['display_color'])
    ax.set_xlabel('Layer', labelpad=10)
    
    if i == 0:
        ax.set_ylabel('Accuracy', labelpad=10)
    
    ax.grid(axis='y', linestyle='-', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_xlim(0, 31)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(np.arange(0, 32, 4))

handles = []
labels = []

for model_id in all_models:
    if model_id in MODEL_MAPPINGS:
        style = MODEL_MAPPINGS[model_id]
        color = style['display_color']
        label = style['display_name']
    else:
        color = 'gray'
        label = model_id
    
    handles.append(Line2D([0], [0], 
                          color=color, 
                          linewidth=3,
                          alpha=1.0))
    labels.append(label)

# Position legend below the plots
lgnd = fig.legend(handles, labels,
                  loc='upper center',
                  bbox_to_anchor=(0.5, 0.025),
                  ncol=len(all_models),
                  frameon=False,
                  handletextpad=0.6,
                  columnspacing=2.0,
                  fontsize=22)

plt.tight_layout()
plt.savefig("figures/layer_accuracies_by_variation.pdf", bbox_inches="tight", dpi=300)
plt.show()



# Steering plot

#### original code: ab_vector_all_formats.ipynb

In [ ]:
var_cps_bootstrap_df = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/var_bootstrap.csv")
base_cps_bootstrap_df = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/base_bootstrap.csv")

In [ ]:
# --- Style ---
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm",
})

variations = ['Consequentialist', 'Emotional', 'Relational']
variation_markers = {'Consequentialist': 's', 'Emotional': 'o', 'Relational': '^'}


def plot_panel(ax, df, variation_col, highlight_side, baselines=None):
    """Plot one panel with three lines (one per variation), weighted only."""
    weighted_data = df[df['vector_type'] == 'weighted']

    for var in variations:
        var_data = weighted_data[weighted_data[variation_col] == var].sort_values('alpha')
        if var_data.empty:
            continue

        color = VARIATION_MAPPINGS[var]['display_color']
        
        if highlight_side == 'left':
            focus    = var_data['alpha'] <= 0
            nonfocus = var_data['alpha'] >= 0
        elif highlight_side == 'right':
            focus    = var_data['alpha'] >= 0
            nonfocus = var_data['alpha'] <= 0
        else:
            focus = np.full(len(var_data), True)
            nonfocus = np.full(len(var_data), False)

        for mask, ci_alpha, line_alpha, mean_alpha in [
            (focus,    0.1, 0.5, 1.0),
            (nonfocus, 0.025, 0.3, 0.35),
        ]:
            seg = var_data[mask]
            if seg.empty:
                continue

            ax.fill_between(
                seg['alpha'], seg['CPS_lower'], seg['CPS_upper'],
                color=color, alpha=ci_alpha, zorder=1
            )
            ax.plot(seg['alpha'], seg['CPS_lower'], color=color, alpha=line_alpha, linewidth=1, zorder=2)
            ax.plot(seg['alpha'], seg['CPS_upper'], color=color, alpha=line_alpha, linewidth=1, zorder=2)
            ax.plot(
                seg['alpha'], seg['CPS_mean'],
                color=color,
                marker=variation_markers[var],
                linewidth=2.5, markersize=8,
                markeredgecolor='black', markeredgewidth=0.7,
                alpha=mean_alpha, zorder=3
            )

            if baselines is not None:
                b = baselines[baselines["variation"] == var].set_index("steering_type")

                for stype, xmin, xmax, side in [("Mute",    0.00, 0.1, "left"),
                                                ("Amplify", 0.90, 1.00, "right")]:
                    if stype not in b.index:
                        continue
                    m = b.loc[stype, "CPS_mean"]
                    a = 0.85 if highlight_side in (None, side) else 0.25
                    ax.axhline(m, xmin=xmin, xmax=xmax, color=color,
                            linewidth=3, solid_capstyle='butt',
                            alpha=a, zorder=4)
                    ax.text(0.035, 1.005, 'Mute\nprompt', transform=ax.transAxes,
                            ha='center', va='bottom', size=11, color='0.35')
                    ax.text(0.965, 1.005, 'Amplify\nprompt', transform=ax.transAxes,
                            ha='center', va='bottom', size=11, color='0.35')
                

    ax.set_xlabel('Alpha', labelpad=8)
    ax.set_ylabel(
        r'$\mathbf{CPS}^{\boldsymbol{(v)}}$ (95% CI)',
        fontsize=20, fontweight='normal'
    )
    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axhline(0, color='gray', linestyle='-', linewidth=2, alpha=0.7)


# --- 1x2 Figure ---
fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_panel(axs[0], var_cps_bootstrap_df,  variation_col='variation',  highlight_side='left')
plot_panel(axs[1], base_cps_bootstrap_df, variation_col='var_vector', highlight_side='right')

axs[0].set_title(
    "Attenuating contextual sensitivity:\n",
    fontsize=20, fontweight="bold", fontstyle="italic", pad=15
)
axs[1].set_title(
    "Inducing contextual sensitivity:\n",
    fontsize=20, fontweight="bold", fontstyle="italic", pad=15
)

fig.tight_layout(rect=[0, 0.08, 1, 1])

# --- Legend ---
handles = [
    Line2D([0], [0], color=VARIATION_MAPPINGS[var]['display_color'], lw=3, marker=variation_markers[var], markersize=7)
    for var in variations
]
fig.legend(
    handles, variations,
    loc='lower center',
    bbox_to_anchor=(0.53, -0.025),
    ncol=3,
    fontsize=20,
    frameon=False
)

plt.show()

In [ ]:
var_cps_bootstrap_df[(var_cps_bootstrap_df['alpha'].isin([-5.0, 0.0, 5.0])) & (var_cps_bootstrap_df['vector_type'] == 'weighted')].round(3)

In [ ]:
# --- Style Updates for a Single-Column Wide Plot ---
plt.rcParams.update({
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 18,
    "figure.figsize": (7, 4.5)  # Ideal for 1-column layout
})

fig, ax = plt.subplots()

# Only plot the attenuation data
plot_panel(ax, var_cps_bootstrap_df, variation_col='variation', highlight_side=None)

# Refined labels and title
# ax.set_title("Attenuating Contextual Sensitivity:\n", 
#              fontsize=16, pad=15)
ax.set_xlabel(r'Steering Coefficient ($\alpha$)', fontsize=20)

# Internal Legend to save space
handles = [
    Line2D([0], [0], color=VARIATION_MAPPINGS[var]['display_color'], 
           lw=2, marker=variation_markers[var], markersize=6)
    for var in variations
]
ax.legend(
    handles, variations,
    loc='upper center',
    bbox_to_anchor=(0.45, -0.29),  # push below plot
    ncol=3,
    frameon=False,
    fontsize=16
)

plt.tight_layout()
plt.show()

## with system prompt baselines

Needs `system_prompt_cps_bootstrap.csv` (from `evaluate/compute_system_prompt_cps.py`), which
needs raw system-prompt-steering response data not included in this repo.

In [ ]:
def plot_panel_with_baselines(ax, df, variation_col, highlight_side, baselines=None):
    """Plot one panel with three lines (one per variation), weighted only."""
    weighted_data = df[df['vector_type'] == 'weighted']

    # --- fixed x-range: data in the middle, empty margins for the prompt stubs ---
    PAD = 1.5                                  # tune this
    ax.set_xlim(-5 - PAD, 5 + PAD)
    ax.set_xticks([-4, -2, 0, 2, 4])

    x0, x1 = ax.get_xlim()
    f_lo = (-5.25 - x0) / (x1 - x0)               # axes fraction where data starts
    f_hi = ( 5.25 - x0) / (x1 - x0)               # axes fraction where data ends

    for var in variations:
        var_data = weighted_data[weighted_data[variation_col] == var].sort_values('alpha')
        if var_data.empty:
            continue

        color = VARIATION_MAPPINGS[var]['display_color']

        if highlight_side == 'left':
            focus    = var_data['alpha'] <= 0
            nonfocus = var_data['alpha'] >= 0
        elif highlight_side == 'right':
            focus    = var_data['alpha'] >= 0
            nonfocus = var_data['alpha'] <= 0
        else:
            focus    = np.full(len(var_data), True)
            nonfocus = np.full(len(var_data), False)

        for mask, ci_alpha, line_alpha, mean_alpha in [
            (focus,    0.1,   0.5, 1.0),
            (nonfocus, 0.025, 0.3, 0.35),
        ]:
            seg = var_data[mask]
            if seg.empty:
                continue

            ax.fill_between(
                seg['alpha'], seg['CPS_lower'], seg['CPS_upper'],
                color=color, alpha=ci_alpha, zorder=1
            )
            ax.plot(seg['alpha'], seg['CPS_lower'], color=color, alpha=line_alpha, linewidth=1, zorder=2)
            ax.plot(seg['alpha'], seg['CPS_upper'], color=color, alpha=line_alpha, linewidth=1, zorder=2)
            ax.plot(
                seg['alpha'], seg['CPS_mean'],
                color=color,
                marker=variation_markers[var],
                linewidth=2.5, markersize=8,
                markeredgecolor='black', markeredgewidth=0.7,
                alpha=mean_alpha, zorder=3
            )

        # --- system-prompt baselines as edge stubs (once per variation) ---
        if baselines is not None:
            b = baselines[baselines["variation"] == var].set_index("steering_type")

            for stype, xmin, xmax, side in [("Mute",    0.0,  f_lo, "left"),
                                            ("Amplify", f_hi, 1.0,  "right")]:
                if stype not in b.index:
                    continue
                a = 0.85 if highlight_side in (None, side) else 0.25
                ax.axhline(
                    b.loc[stype, "CPS_mean"],
                    xmin=xmin, xmax=xmax,
                    color=color, linewidth=3, solid_capstyle='butt',
                    alpha=a, zorder=4,
                )

    # --- stub labels (once per panel) ---
    if baselines is not None:
        ax.text(f_lo / 2, 1.005, 'Mute\nprompt', transform=ax.transAxes,
                ha='center', va='bottom', size=11, color='0.35')
        ax.text((1 + f_hi) / 2, 1.005, 'Amplify\nprompt', transform=ax.transAxes,
                ha='center', va='bottom', size=11, color='0.35')
    ax.set_yticks([-0.1, 0.0, 0.1, 0.2])

    ax.set_xlabel('Alpha', labelpad=8)
    ax.set_ylabel(
        r'$\mathbf{CPS}^{\boldsymbol{(v)}}$ (95% CI)',
        fontsize=20, fontweight='normal'
    )
    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axhline(0, color='gray', linestyle='-', linewidth=2, alpha=0.7)

In [ ]:
system_prompt_baselines_df = pd.read_csv(PATH_ANALYSIS / "system_prompt_cps_bootstrap.csv")

In [ ]:
# --- Style Updates for a Single-Column Wide Plot ---
plt.rcParams.update({
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 18,
    "figure.figsize": (7, 4.5)  # Ideal for 1-column layout
})

fig, ax = plt.subplots()

# Only plot the attenuation data
plot_panel_with_baselines(ax, var_cps_bootstrap_df, variation_col='variation', highlight_side=None, baselines=system_prompt_baselines_df)

# Refined labels and title
# ax.set_title("Attenuating Contextual Sensitivity:\n", 
#              fontsize=16, pad=15)
ax.set_xlabel(r'Steering Coefficient ($\alpha$)', fontsize=20)

# Internal Legend to save space
handles = [
    Line2D([0], [0], color=VARIATION_MAPPINGS[var]['display_color'], 
           lw=2, marker=variation_markers[var], markersize=6)
    for var in variations
]
ax.legend(
    handles, variations,
    loc='upper center',
    bbox_to_anchor=(0.45, -0.29),  # push below plot
    ncol=3,
    frameon=False,
    fontsize=16
)

plt.tight_layout()
plt.savefig("figures/steering_attenuation_only_w_system_prompt.pdf", bbox_inches='tight')
plt.show()

# Steering distribution plot

#### original code: ab_vector_all_formats.ipynb

In [ ]:
base_cps = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/base.csv")
var_cps = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/var.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import numpy as np

# --- Style ---
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

data_all = var_cps[(var_cps['vector_type'] == 'weighted')].copy()
data_all['alpha'] = data_all['alpha'].astype(float)
alphas = sorted(data_all['alpha'].unique())
n = len(alphas)

x_min = data_all['CPS'].min() - 0.05
x_max = data_all['CPS'].max() + 0.05

fig = plt.figure(figsize=(20, 8), dpi=300)
outer_gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.15)

for col_idx, variation_name in enumerate(variations):
    inner_gs = gridspec.GridSpecFromSubplotSpec(n, 1, subplot_spec=outer_gs[col_idx], hspace=-0.15)
    axs = [fig.add_subplot(inner_gs[i]) for i in range(n)]

    data = data_all[data_all['variation'] == variation_name]
    base_color = mcolors.to_rgb(VARIATION_MAPPINGS[variation_name]['display_color'])

    # Build palette: faded (light) -> full color
    def make_palette(base_color, alphas):
        color_map = {}
        for a in alphas:
            if a > 0:
                color_map[a] = (*base_color, 0.35)  # faded
            else:
                color_a = (*base_color, 1.0)  # full opacity
                color_map[a] = color_a
        return color_map

    color_map = make_palette(base_color, alphas)


    for i, alpha_val in enumerate(alphas):
        ax = axs[i]
        subset = data[data['alpha'] == alpha_val]['CPS']
        color = color_map[alpha_val]

        fill_alpha = 0.35 if alpha_val > 0 else 1.0

        sns.kdeplot(
            subset, ax=ax,
            bw_adjust=0.5,
            fill=True, alpha=fill_alpha,
            color=color,
            linewidth=0,
            clip=(x_min, x_max)
        )
        sns.kdeplot(
            subset, ax=ax,
            bw_adjust=0.5,
            fill=False,
            color='white',
            linewidth=2,
            alpha=fill_alpha,  # also fade the white outline
            clip=(x_min, x_max)
        )

        mean_val = subset.mean()
        y_top = ax.get_ylim()[1] * 0.85
        ax.plot(
            [mean_val, mean_val],
            [0, y_top],              # explicitly bounded to this axis's height
            lw=2,
            color='black',
            linestyle=':',
            alpha=0.9,
            zorder=4,
            clip_on=True             # clip to this axis only
        )

        ax.axvline(0, lw=2, color='black', linestyle='--', clip_on=True)

        if col_idx == 0:
            ax.text(
                -0.12, 0.2,
                f"α = {alpha_val:.1f}",
                fontweight="bold",
                color="black",
                ha="right", va="center",
                transform=ax.transAxes,
                clip_on=False,
                fontsize=17,
                fontfamily="serif"
            )

        ax.patch.set_alpha(0)
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.set_xlabel("")        # suppress per-row xlabel always
        ax.set_xlim(x_min, x_max)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)

        if i < n - 1:
            ax.set_xticks([])
            ax.spines['bottom'].set_visible(False)
        else:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_color("#000000")
            ax.spines['bottom'].set_linewidth(1.0)
            ax.tick_params(axis='x', labelsize=16)
            ax.set_xlabel(f"Scenario CPS Distribution", labelpad=8, fontsize=18)

        if i == 0:
            ax.set_title(
                variation_name,
                fontsize=20,
                pad=10,
                color=VARIATION_MAPPINGS[variation_name]['display_color']
            )

#plt.savefig(f"figures/steered_cps_var_distributions.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
min(data_all['CPS']), max(data_all['CPS'])

In [ ]:
# --- Style ---
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

data_all = base_cps[(base_cps['vector_type'] == 'weighted')].copy()
data_all['alpha'] = data_all['alpha'].astype(float)
#alphas = sorted(data_all['alpha'].unique())
alphas = sorted([a for a in data_all['alpha'].unique() if a != 0.0])

n = len(alphas)

x_min = data_all['CPS'].min() - 0.05
x_max = data_all['CPS'].max() + 0.05

fig = plt.figure(figsize=(20, 8), dpi=300)
outer_gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.15)

for col_idx, variation_name in enumerate(variations):
    inner_gs = gridspec.GridSpecFromSubplotSpec(n, 1, subplot_spec=outer_gs[col_idx], hspace=-0.15)
    axs = [fig.add_subplot(inner_gs[i]) for i in range(n)]

    data = data_all[data_all['var_vector'] == variation_name]
    base_color = mcolors.to_rgb(VARIATION_MAPPINGS[variation_name]['display_color'])

    def make_palette(base_color, alphas):
        color_map = {}
        for a in alphas:
            if a < 0:
                color_map[a] = (*base_color, 0.1) 
            else:
                color_a = (*base_color, 1.0)
                color_map[a] = color_a
        return color_map

    color_map = make_palette(base_color, alphas)


    for i, alpha_val in enumerate(alphas):
        ax = axs[i]
        subset = data[data['alpha'] == alpha_val]['CPS']
        color = color_map[alpha_val]

        fill_alpha = 0.35 if alpha_val < 0 else 1.0

        sns.kdeplot(
            subset, ax=ax,
            bw_adjust=0.5,
            fill=True, alpha=fill_alpha,
            color=color,
            linewidth=0,
            clip=(x_min, x_max)
        )
        sns.kdeplot(
            subset, ax=ax,
            bw_adjust=0.5,
            fill=False,
            color='white',
            linewidth=2,
            alpha=fill_alpha,
            clip=(x_min, x_max)
        )

        mean_val = subset.mean()
        y_top = ax.get_ylim()[1] * 0.85
        ax.plot(
            [mean_val, mean_val],
            [0, y_top], 
            lw=2,
            color='black',
            linestyle=':',
            alpha=0.9,
            zorder=4,
            clip_on=True 
        )

        ax.axvline(0, lw=2, color='black', linestyle='--', clip_on=True)

        if col_idx == 0:
            ax.text(
                -0.12, 0.2,
                f"α = {alpha_val:.1f}",
                fontweight="bold",
                color="black",
                ha="right", va="center",
                transform=ax.transAxes,
                clip_on=False,
                fontsize=17,
                fontfamily="serif"
            )

        ax.patch.set_alpha(0)
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.set_xlabel("")
        ax.set_xlim(x_min, x_max)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)

        if i < n - 1:
            ax.set_xticks([])
            ax.spines['bottom'].set_visible(False)
        else:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_color("#000000")
            ax.spines['bottom'].set_linewidth(1.0)
            ax.tick_params(axis='x', labelsize=16)
            ax.set_xlabel(f"Scenario CPS Distribution", labelpad=8, fontsize=18)

        if i == 0:
            ax.set_title(
                variation_name,
                fontsize=20,
                pad=10,
                color=VARIATION_MAPPINGS[variation_name]['display_color']
            )

# --- Single shared x-label ---
# fig.text(
#     0.5, -0.02,
#     "Individual Scenario CPS",
#     ha='center', va='center',
#     fontsize=20,
#     fontweight='bold',
#     fontfamily='serif'
# )
#plt.savefig(f"figures/steered_cps_base_distributions.pdf", bbox_inches="tight", dpi=300)
plt.show()

# Steering configuration plot

#### original code: activation_engineering_configuration_plot.ipynb

In [ ]:
cps_config_results = json.load(open(PATH_ANALYSIS / "ab_vector_steering_dfs/cps_config_data.json", "r"))

In [ ]:
# --- Style ---
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

GROUPS = [
    ("AB + Compare + Repeat",  "#E66101", "D", ["ab_comp_rep_vector_weighted",   "ab_comp_rep_vector_unweighted"]),
    ("AB",             "#4A90D9", "o", ["ab_vector_weighted",               "ab_vector_unweighted"]),
    ("Compare",        "#7B6FD6", "^", ["comp_vector_weighted",             "comp_vector_unweighted"]),
    ("Repeat",         "#C06C84", "P", ["rep_vector_weighted",              "rep_vector_unweighted"]),
    ("AB + Compare",     "#2AAA6F", "s", ["ab_comp_vector_weighted",          "ab_comp_vector_unweighted"]),
    ("AB + Repeat",      "#D4A24C", "X", ["ab_rep_vector_weighted",           "ab_rep_vector_unweighted"]),
    ("Compare + Repeat", "#888888", "v", ["comp_rep_vector_weighted",         "comp_rep_vector_unweighted"]),
]

WEIGHT_STYLES = {
    "weighted":   {"linestyle": "-"},
    "unweighted": {"linestyle": ":"},
}

def get_weight_tag(key):
    return "weighted" if key.endswith("weighted)") or key.endswith("_weighted") else "unweighted"

key_to_style = {}
for group_label, color, marker, keys in GROUPS:
    for k in keys:
        if k in cps_config_results:
            w = get_weight_tag(k)
            key_to_style[k] = {
                "color":     color,
                "linestyle": WEIGHT_STYLES[w]["linestyle"],
                "marker":    marker,
            }

# row_titles = [
#     r"Muting contextual variations: $\tilde{h}_L(v(x_i)) = h_L(v(z(x_i))) + \alpha \cdot \mathbf{s}_L^{(v)}$",
#     r"Simulating contextual variations: $\tilde{h}_L(x_i) = h_L(z(x_i)) + \alpha \cdot \mathbf{s}_L^{(v)}$"
# ]

row_titles = [
    r"Attenuating contextual sensitivity: ",
    r"Inducing contextual sensitivity: "
]


def make_plot(data_key, fig_title, output_path):
    fig, axs = plt.subplots(2, 3, figsize=(22, 12), sharey='row', sharex=False)

    all_keys = list(cps_config_results.keys())

    for row_idx, cps_type in enumerate(['var_cps', 'base_cps']):
        for col_idx, variation in enumerate(variations):
            ax = axs[row_idx, col_idx]

            for vec_key in all_keys:
                if vec_key not in key_to_style:
                    continue
                vec_data = cps_config_results[vec_key][data_key][cps_type]
                if variation not in vec_data:
                    continue

                style = key_to_style[vec_key]
                y_vals = np.array(vec_data[variation])
                x_vals = np.linspace(-5.0, 5.0, len(y_vals))

                if cps_type == 'var_cps':
                    focus_mask    = x_vals <= 0
                    nonfocus_mask = x_vals >= 0
                else:
                    focus_mask    = x_vals >= 0
                    nonfocus_mask = x_vals <= 0

                for mask, alpha_val in [(focus_mask, 1.0), (nonfocus_mask, 0.34)]:
                    x_seg = x_vals[mask]
                    y_seg = y_vals[mask]
                    if len(x_seg) == 0:
                        continue
                    ax.plot(
                        x_seg, y_seg,
                        color=style["color"],
                        linestyle=style["linestyle"],
                        marker=style["marker"],
                        linewidth=2.2,
                        markersize=7,
                        markeredgecolor='black',
                        markeredgewidth=0.6,
                        alpha=alpha_val,
                        zorder=3,
                    )

            if row_idx == 0:
                ax.set_title(variation, fontsize=22, pad=60, color=VARIATION_MAPPINGS[variation]['display_color'])

            if col_idx == 0:
                ax.set_ylabel(r'$\mathbf{CPS}^{\boldsymbol{(v)}}$', fontsize=20, fontweight='normal')

            ax.set_xlabel('Alpha', labelpad=8)
            ax.grid(axis="y", linestyle="-", alpha=0.2)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.axhline(0, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)

    # Row titles
    # fig.text(0.525, 0.965, row_titles[0], ha="center", va="top",
    #          fontsize=22, fontweight="bold", fontstyle="italic")
    # fig.text(0.525, 0.52, row_titles[1], ha="center", va="top",
    #          fontsize=22, fontweight="bold", fontstyle="italic")
    fig.text(0.42125, 0.92, row_titles[0], ha="center", va="top",
             fontsize=22, fontweight="bold", fontstyle="italic")
    fig.text(0.42125, 0.51, row_titles[1], ha="center", va="top",
             fontsize=22, fontweight="bold", fontstyle="italic")


    fig.tight_layout(rect=[0, 0.13, 1, 0.95])
    fig.subplots_adjust(top=0.875, bottom=0.18, hspace=0.40)

    n_groups = len(GROUPS)

    legend_bottom = -0.06
    legend_top    = legend_bottom + 0.055
    header_y      = legend_top   + 0.045

    x_start = 0.125
    x_end   = 0.9
    col_positions = [x_start + i * (x_end - x_start) / (n_groups - 1) for i in range(n_groups)]

    fig.text(0.53, legend_top + 0.085, "Steering Vector Source",
             ha='center', va='center',
             fontsize=24, fontweight='bold', color='black', fontstyle='italic',
             transform=fig.transFigure)
    

    for i, (group_label, color, marker, keys) in enumerate(GROUPS):
        x = col_positions[i]

        fig.text(x, header_y, group_label,
                 ha='center', va='center',
                 fontsize=22, fontweight='bold', color='black',
                 transform=fig.transFigure)

        for row_y, linestyle, label in [
            (legend_top,    '-',  'Weighted'),
            (legend_bottom, ':', 'Unweighted'),
        ]:
            ax_inset = fig.add_axes([x - 0.045, row_y - 0.012, 0.06, 0.024])
            ax_inset.set_axis_off()
            ax_inset.plot([0.05, 0.55], [0.5, 0.5],
                          color=color, linestyle=linestyle,
                          linewidth=2.2, marker=marker,
                          markersize=10, markeredgecolor='black',
                          markeredgewidth=0.6,
                          transform=ax_inset.transAxes,
                          clip_on=False)
            fig.text(x + 0.0, row_y, label,
                     ha='left', va='center',
                     fontsize=20, color='black',
                     transform=fig.transFigure)

    #plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()


make_plot('last_token',    'Last Token Position', 'figures/cps_last_token_clean.pdf')
make_plot('all_positions', 'All Positions',        'figures/cps_all_positions_clean.pdf')

# Benchmark accuracy plot

#### original code: benchmark_analysis.ipynb

Needs `benchmarks/results/{MMLU,HELLASWAG,ETHICS}_results.csv` (from
`evaluate/compute_benchmark_results.py`), which needs raw lm-eval-harness JSON output not
included in this repo.

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

variations = ['Consequentialist', 'Emotional', 'Relational']
variation_markers = {'Consequentialist': 's', 'Emotional': 'o', 'Relational': '^'}

def load_benchmark_results(benchmark):
    path = f"{PATH_ANALYSIS}/benchmarks/results/{benchmark}_results.csv"
    data = pd.read_csv(path)
    if data.empty:
        print(f"Warning: No data found for {benchmark} at {path}")
        return None
    return pd.DataFrame({
        'benchmark': data['benchmark'],
        'variation': data['variation'],
        'alpha': data['alpha'],
        'macro_avg': data['macro_avg']
    })

mmlu_data = load_benchmark_results("MMLU")

hellaswag_data = load_benchmark_results("HELLASWAG")

ethics_data = load_benchmark_results("ETHICS")

benchmark_dfs = {
    'MMLU':      mmlu_data,
    'HellaSwag': hellaswag_data,
    'ETHICS':    ethics_data,
}

ax_y_limits = {
    'MMLU': (0.655, 0.69),
    'HellaSwag': (0.5605, 0.58),
    'ETHICS': (0.61, 0.66),
}

def plot_panel(ax, df, title):
    for var in variations:
        var_data = df[df['variation'] == var].sort_values('alpha')
        if var_data.empty:
            continue

        color  = VARIATION_MAPPINGS[var]['display_color']
        marker = variation_markers[var]

        ax.plot(
            var_data['alpha'], var_data['macro_avg'],
            color=color,
            marker=marker,
            linewidth=2.5, markersize=8,
            markeredgecolor='black', markeredgewidth=0.7,
            zorder=3,
            label=var,
        )

    ax.set_xlabel('Alpha', labelpad=8)
    ax.set_title(title, fontsize=26, fontweight='bold', pad=15)
    ax.grid(axis='y', linestyle='-', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)

fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharey=False, constrained_layout=True)

for ax, (benchmark, df) in zip(axs, benchmark_dfs.items()):
    plot_panel(ax, df, title=benchmark)
    ax.set_ylim(ax_y_limits[benchmark])
    ax.set_xticks([-4.0, -2.0, 0.0, 2.0, 4.0])

axs[0].set_ylabel('Accuracy', labelpad=10)

fig.tight_layout()

handles = [
    Line2D([0], [0],
           color=VARIATION_MAPPINGS[var]['display_color'],
           lw=3,
           marker=variation_markers[var],
           markersize=7,
           markeredgecolor='black', markeredgewidth=0.7)
    for var in variations
]
fig.legend(
    handles, variations,
    loc='lower center',
    bbox_to_anchor=(0.525, -0.125),
    ncol=3,
    fontsize=22,
    frameon=False,
)

#plt.savefig("figures/benchmarks.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Style ---
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

variations = ['Consequentialist', 'Emotional', 'Relational']
variation_markers = {'Consequentialist': 's', 'Emotional': 'o', 'Relational': '^'}

# --- Data ---
def load_benchmark_results(benchmark):
    path = f"{PATH_ANALYSIS}/benchmarks/results/{benchmark}_results.csv"
    data = pd.read_csv(path)
    if data.empty:
        print(f"Warning: No data found for {benchmark} at {path}")
        return None
    return pd.DataFrame({
        'benchmark': data['benchmark'],
        'variation': data['variation'],
        'alpha': data['alpha'],
        'macro_avg': data['macro_avg']
    })


mmlu_data = load_benchmark_results("MMLU")

hellaswag_data = load_benchmark_results("HELLASWAG")

ethics_data = load_benchmark_results("ETHICS")

benchmark_dfs = {
    'MMLU':      mmlu_data,
    'HellaSwag': hellaswag_data,
    'ETHICS':    ethics_data,
}

ax_y_limits = {
    'MMLU': (0.655, 0.69),
    'HellaSwag': (0.5605, 0.58),
    'ETHICS': (0.61, 0.66),
}

def plot_panel(ax, df, title):
    for var in variations:
        var_data = df[df['variation'] == var].sort_values('alpha')
        if var_data.empty:
            continue

        color  = VARIATION_MAPPINGS[var]['display_color']
        marker = variation_markers[var]

        ax.plot(
            var_data['alpha'], var_data['macro_avg'],
            color=color,
            marker=marker,
            linewidth=2.5, markersize=8,
            markeredgecolor='black', markeredgewidth=0.7,
            zorder=3,
            label=var,
        )

    ax.set_xlabel('Alpha', labelpad=8)
    ax.set_title(title, fontsize=26, fontweight='bold', pad=15)
    ax.grid(axis='y', linestyle='-', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)


fig, axs = plt.subplots(1, 3, figsize=(16, 3.5), sharey=False, constrained_layout=True)

for ax, (benchmark, df) in zip(axs, benchmark_dfs.items()):
    plot_panel(ax, df, title=benchmark)
    ax.set_ylim(ax_y_limits[benchmark])
    ax.set_xticks([-4.0, -2.0, 0.0, 2.0, 4.0])

axs[0].set_ylabel('Accuracy', labelpad=10)

fig.tight_layout()


handles = [
    Line2D([0], [0],
           color=VARIATION_MAPPINGS[var]['display_color'],
           lw=3,
           marker=variation_markers[var],
           markersize=7,
           markeredgecolor='black', markeredgewidth=0.7)
    for var in variations
]
fig.legend(
    handles, variations,
    loc='lower center',
    bbox_to_anchor=(0.525, -0.125),
    ncol=3,
    fontsize=22,
    frameon=False,
)

plt.savefig("figures/benchmarks_shallow.pdf", dpi=300, bbox_inches='tight')
plt.show()